# WFS-mimic deviation covariance — reader

**Author:** Aaron Roodman &nbsp;|&nbsp; **Purpose:** read and visualize the FAM-derived corner-WFS
covariance products written by the `wfs_mimic` Snakemake step (`run_wfs_mimic.py`).

Thin companion to that step: it loads the parquet matrices and plots them; the heavy
SVD / DOF-recovery / FWHM exploration stays in `study_wfs_mimic.ipynb`.

### Inputs (per `output/<param_set>/<mi_name>/wfs_mimic/`)
| file | contents |
|---|---|
| `wfs_mimic_cov84.parquet` | 84×84 cross-corner covariance + correlation of the deviation (measured − OCS/CCS intrinsic), sampled over images. Labels `Z{j}_c{0..3}`. |
| `wfs_mimic_cov21.parquet` | 21×21 single-corner covariance (mean of the four diagonal corner blocks). |
| `wfs_mimic_cov_bins.parquet` | per rotator-subset 84×84 (tidy: one row per (subset, label)). |

The deviation is the wavefront error a WFS would feed to OFC; the cross-corner blocks
(`Z_{j,c}·Z_{j',c'}`) show how the four corner estimates co-vary. `.meta` carries the
corner positions, `n_images`, `rank`, `rotator_keep`, and the wedge geometry.

## 1. Parameters

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

# directory holding the wfs_mimic products
param_set = "fam_danish_1_0_wep17_3_0_bin2x"
mi_name   = "pathA_50_34_i_5rot"
wfs_dir   = Path(f"output/{param_set}/{mi_name}/wfs_mimic")

## 2. Read the matrices

In [ ]:
def _matrix(tbl, col):
    """Stack a row-wise list column back into a square (N, N) array."""
    return np.vstack([np.asarray(r, dtype=float) for r in tbl[col]])

t84 = Table.read(str(wfs_dir / "wfs_mimic_cov84.parquet"), format="parquet")
t21 = Table.read(str(wfs_dir / "wfs_mimic_cov21.parquet"), format="parquet")
cov84, corr84 = _matrix(t84, "cov"), _matrix(t84, "corr")
cov21, corr21 = _matrix(t21, "cov"), _matrix(t21, "corr")
labels84 = list(t84.meta["labels"]); labels21 = list(t21.meta["labels"])
noll = [int(j) for j in t84.meta["noll"]]; nZk = len(noll)

print(f"84x84: {cov84.shape}, 21x21: {cov21.shape}")
for k in ("n_images", "rank", "quantity", "cov_units", "rotator_keep",
          "wfs_inner_radius_deg", "wfs_outer_radius_deg", "delta_deg",
          "min_donuts_per_wedge"):
    print(f"  {k}: {t84.meta.get(k)}")
if t84.meta.get("rank", cov84.shape[0]) < cov84.shape[0]:
    print(f"  NOTE: 84x84 is rank-deficient ({t84.meta['rank']}/{cov84.shape[0]}) "
          f"-- fewer than 84 images; correlation is usable, covariance is singular.")

## 3. Covariance / correlation heatmaps

In [ ]:
def _heat(ax, A, title, labels, unit, diverging=True):
    finite = A[np.isfinite(A)]
    vmax = (float(np.nanpercentile(np.abs(finite), 99)) if finite.size else 1.0) or 1e-9
    im = ax.imshow(A, cmap="RdBu_r", vmin=-vmax, vmax=vmax) if diverging else \
         ax.imshow(A, cmap="viridis", vmin=0, vmax=vmax)
    ax.set_title(title, fontsize=10)
    step = max(1, len(labels) // 28)
    t = list(range(0, len(labels), step))
    ax.set_xticks(t); ax.set_yticks(t)
    ax.set_xticklabels([labels[i] for i in t], rotation=90, fontsize=5)
    ax.set_yticklabels([labels[i] for i in t], fontsize=5)
    plt.colorbar(im, ax=ax, shrink=0.8, label=unit)

fig, ax = plt.subplots(2, 2, figsize=(15, 14), layout="constrained")
_heat(ax[0, 0], cov84, "84x84 covariance", labels84, "$\mu$m$^2$")
_heat(ax[0, 1], corr84, "84x84 correlation", labels84, "r")
_heat(ax[1, 0], cov21, "21x21 covariance (corner-mean)", labels21, "$\mu$m$^2$")
_heat(ax[1, 1], corr21, "21x21 correlation (corner-mean)", labels21, "r")
fig.suptitle(f"WFS-mimic deviation covariance  ({t84.meta['n_images']} images)", fontsize=13)
plt.show()

## 4. Cross-corner structure
For each Zernike, the four corner estimates form a 4×4 sub-block of the 84×84 correlation
matrix. Its mean off-diagonal is how strongly that Zernike co-varies *between* corners
(common-mode telescope drift → high; corner-local noise → low).

In [ ]:
# 84x84 is corner-major: index(c, ji) = c*nZk + ji
mean_offdiag = []
for ji in range(nZk):
    idx = [c * nZk + ji for c in range(4)]
    sub = corr84[np.ix_(idx, idx)]
    off = sub[~np.eye(4, dtype=bool)]
    mean_offdiag.append(float(np.nanmean(off)))

fig, ax = plt.subplots(figsize=(10, 3.5), layout="constrained")
ax.bar([f"Z{j}" for j in noll], mean_offdiag, color="steelblue")
ax.axhline(0, color="k", lw=0.5)
ax.set_ylabel("mean corner-corner r"); ax.set_ylim(-1, 1)
ax.set_title("Between-corner correlation per Zernike")
ax.tick_params(axis="x", rotation=90, labelsize=7)
plt.show()

## 5. Per rotator-subset summary

In [ ]:
tb = Table.read(str(wfs_dir / "wfs_mimic_cov_bins.parquet"), format="parquet")
subsets = []
for name in dict.fromkeys([str(s) for s in tb["subset"]]):
    m = np.array([str(s) == name for s in tb["subset"]])
    subsets.append((name, int(np.asarray(tb["n_images"])[m][0]),
                    int(np.asarray(tb["rank"])[m][0])))
print(f"{'subset':<16}{'n_images':>9}{'rank':>7}")
for name, n, rk in subsets:
    print(f"{name:<16}{n:>9}{rk:>7}")